# 03 - Dataset Preview, Quality Verification & PyTorch Loader Test

This notebook verifies the generated dataset before training GAN models:
- Visualizes paired False Color Composites (Clear Ground Truth vs Synthetic Cloudy Input)
- Checks pixel ranges and band distributions
- Tests the PyTorch `LISSPairDataset` and `DataLoader`


In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from scripts.preview_patches import to_fcc, preview_dataset
from scripts.dataset_loader import LISSPairDataset, get_dataloaders, TORCH_AVAILABLE

DATA_DIR = PROJECT_ROOT / "Data"
CLEAR_DIR = DATA_DIR / "Clear_Patches"
CLOUDY_DIR = DATA_DIR / "Cloudy_Patches"

clear_files = sorted(list(CLEAR_DIR.glob("*.npy")))
cloudy_files = sorted(list(CLOUDY_DIR.glob("*.npy")))
print(f"Clear patches:  {len(clear_files)}")
print(f"Cloudy patches: {len(cloudy_files)}")


### 1. Side-by-Side FCC Patch Visualizations


In [ ]:
# Preview 4 random patch pairs
num_samples = 4
indices = np.random.choice(len(clear_files), size=num_samples, replace=False)

fig, axes = plt.subplots(num_samples, 2, figsize=(10, 4 * num_samples))
for row, idx in enumerate(indices):
    clear_patch = np.load(clear_files[idx])
    cloudy_patch = np.load(cloudy_files[idx])
    
    axes[row, 0].imshow(to_fcc(clear_patch))
    axes[row, 0].set_title(f"Clear Ground Truth [{clear_files[idx].name}] (FCC: NIR, Red, Green)")
    axes[row, 0].axis("off")
    
    axes[row, 1].imshow(to_fcc(cloudy_patch))
    axes[row, 1].set_title(f"Synthetic Cloudy Input [{cloudy_files[idx].name}]")
    axes[row, 1].axis("off")

plt.tight_layout()
plt.show()


### 2. Pixel Value Distribution Across Bands


In [ ]:
sample_clear = np.load(clear_files[indices[0]])
sample_cloudy = np.load(cloudy_files[indices[0]])

bands = ["B2 (Green)", "B3 (Red)", "B4 (NIR)"]
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for i, bname in enumerate(bands):
    axes[i].hist(sample_clear[i].ravel(), bins=50, alpha=0.6, label="Clear", color="green")
    axes[i].hist(sample_cloudy[i].ravel(), bins=50, alpha=0.6, label="Cloudy", color="blue")
    axes[i].set_title(f"Band: {bname}")
    axes[i].set_xlabel("Normalized Pixel Value [0, 1]")
    axes[i].set_ylabel("Frequency")
    axes[i].legend()

plt.suptitle("Pixel Intensity Distributions (Clear vs Cloudy)", fontsize=13)
plt.tight_layout()
plt.show()


### 3. Test PyTorch Dataset & DataLoader


In [ ]:
if not TORCH_AVAILABLE:
    print("PyTorch not installed in environment. Run `pip install torch` to test DataLoader.")
else:
    train_loader, val_loader = get_dataloaders(CLEAR_DIR, CLOUDY_DIR, batch_size=8)
    batch_cloudy, batch_clear = next(iter(train_loader))
    print(f"Batch Cloudy Tensor: shape={batch_cloudy.shape}, range=[{batch_cloudy.min():.2f}, {batch_cloudy.max():.2f}]")
    print(f"Batch Clear Tensor:  shape={batch_clear.shape}, range=[{batch_clear.min():.2f}, {batch_clear.max():.2f}]")
